<!-- project-navigation -->
# 08_data_understandin_credit_card_balance
Role: Exploration and data preparation.

See [notebook guide](README.md) and [model comparison](../reports/model_comparison.md). Historical code and outputs are retained; run with the project root as working directory.


In [6]:
import pandas as pd
import numpy as np


# ============================================================
# 1) Chargement de la table
# ============================================================

credit_card = pd.read_csv(
    "data/raw/credit_card_balance.csv"
)


print("========== DIMENSION INITIALE ==========")
print(credit_card.shape)



# ============================================================
# 2) Informations générales
# ============================================================

print("\n========== INFORMATIONS GENERALES ==========")
print(credit_card.info())


print("\n========== VARIABLES ==========")
print(credit_card.columns.tolist())



# ============================================================
# 3) Valeurs manquantes avant traitement
# ============================================================

missing = credit_card.isna().sum()

missing = missing[missing > 0]


missing_table = pd.DataFrame({
    "Nombre_manquant": missing,
    "Pourcentage": (missing / len(credit_card))*100
}).sort_values(
    by="Pourcentage",
    ascending=False
)


print("\n========== VALEURS MANQUANTES ==========")
display(missing_table)



# ============================================================
# 4) Analyse statut crédit
# ============================================================

print("\n========== STATUS CREDIT ==========")

print(
    credit_card["NAME_CONTRACT_STATUS"]
    .value_counts()
)



# ============================================================
# 5) Traitement des valeurs manquantes
# ============================================================

print("\nTraitement des valeurs manquantes...")


# Variables où absence = aucune activité
zero_fill = [
    "AMT_DRAWINGS_ATM_CURRENT",
    "AMT_DRAWINGS_CURRENT",
    "AMT_DRAWINGS_OTHER_CURRENT",
    "AMT_DRAWINGS_POS_CURRENT",
    "AMT_PAYMENT_CURRENT",
    "AMT_PAYMENT_TOTAL_CURRENT",
    "SK_DPD",
    "SK_DPD_DEF"
]


for col in zero_fill:
    if col in credit_card.columns:
        credit_card[col] = credit_card[col].fillna(0)



# Variables financières restantes :
# remplacement par médiane

numeric_cols = credit_card.select_dtypes(
    include=np.number
).columns


for col in numeric_cols:
    
    if col not in ["SK_ID_CURR", "SK_ID_PREV"]:
        
        credit_card[col] = credit_card[col].fillna(
            credit_card[col].median()
        )



print("\nValeurs manquantes après traitement :")
print(
    credit_card.isna().sum().sum()
)



# ============================================================
# 6) Création des variables comportementales client
# ============================================================

print("\nCréation des variables métier...")


credit_features = credit_card.groupby(
    "SK_ID_CURR"
).agg(


    # nombre total d'observations mensuelles
    NB_CREDIT_CARD_RECORDS = (
        "SK_ID_PREV",
        "count"
    ),


    # nombre de cartes différentes
    NB_CREDIT_CARDS = (
        "SK_ID_PREV",
        "nunique"
    ),


    # solde moyen
    AVG_BALANCE = (
        "AMT_BALANCE",
        "mean"
    ),


    # solde maximal
    MAX_BALANCE = (
        "AMT_BALANCE",
        "max"
    ),


    # limite moyenne
    AVG_CREDIT_LIMIT = (
        "AMT_CREDIT_LIMIT_ACTUAL",
        "mean"
    ),


    # paiement moyen
    AVG_PAYMENT = (
        "AMT_PAYMENT_TOTAL_CURRENT",
        "mean"
    ),


    # montant total payé
    TOTAL_PAYMENT = (
        "AMT_PAYMENT_TOTAL_CURRENT",
        "sum"
    ),


    # nombre de mois avec retard
    NB_DELAY_MONTHS = (
        "SK_DPD",
        lambda x: (x > 0).sum()
    ),


    # retard maximal
    MAX_DPD = (
        "SK_DPD",
        "max"
    ),


    MAX_DPD_DEF = (
        "SK_DPD_DEF",
        "max"
    )



).reset_index()



# ============================================================
# 7) Variables dérivées
# ============================================================


# utilisation de la carte

credit_features["CREDIT_UTILIZATION_RATIO"] = (

    credit_features["AVG_BALANCE"]

    /

    credit_features["AVG_CREDIT_LIMIT"]

)



# ratio de retard

credit_features["DPD_RATIO"] = (

    credit_features["NB_DELAY_MONTHS"]

    /

    credit_features["NB_CREDIT_CARD_RECORDS"]

)



# remplacement valeurs infinies

credit_features.replace(
    [np.inf, -np.inf],
    np.nan,
    inplace=True
)



# traitement final des valeurs manquantes

credit_features.fillna(
    0,
    inplace=True
)



# ============================================================
# 8) Vérification finale
# ============================================================


print("\n========== TABLE FINALE ==========")

print(
    credit_features.shape
)


print("\nValeurs manquantes :")

print(
    credit_features.isna().sum().sum()
)



print("\nAperçu :")

display(
    credit_features.head()
)



# ============================================================
# 9) Sauvegarde
# ============================================================

credit_features.to_csv(
    "data/processed/credit_card_features.csv",
    index=False
)


print(
    "\nFichier sauvegardé : credit_card_features.csv"
)

========== DIMENSION INITIALE ==========
(3840312, 23)

========== INFORMATIONS GENERALES ==========
<class 'pandas.DataFrame'>
RangeIndex: 3840312 entries, 0 to 3840311
Data columns (total 23 columns):
 #   Column                      Dtype  
---  ------                      -----  
 0   SK_ID_PREV                  int64  
 1   SK_ID_CURR                  int64  
 2   MONTHS_BALANCE              int64  
 3   AMT_BALANCE                 float64
 4   AMT_CREDIT_LIMIT_ACTUAL     int64  
 5   AMT_DRAWINGS_ATM_CURRENT    float64
 6   AMT_DRAWINGS_CURRENT        float64
 7   AMT_DRAWINGS_OTHER_CURRENT  float64
 8   AMT_DRAWINGS_POS_CURRENT    float64
 9   AMT_INST_MIN_REGULARITY     float64
 10  AMT_PAYMENT_CURRENT         float64
 11  AMT_PAYMENT_TOTAL_CURRENT   float64
 12  AMT_RECEIVABLE_PRINCIPAL    float64
 13  AMT_RECIVABLE               float64
 14  AMT_TOTAL_RECEIVABLE        float64
 15  CNT_DRAWINGS_ATM_CURRENT    float64
 16  CNT_DRAWINGS_CURRENT        int64  
 17  CNT_DRAWINGS_

,Nombre_manquant,Pourcentage
AMT_PAYMENT_CURRENT,767988,19.998063
AMT_DRAWINGS_ATM_CURRENT,749816,19.524872
AMT_DRAWINGS_OTHER_CURRENT,749816,19.524872
AMT_DRAWINGS_POS_CURRENT,749816,19.524872
CNT_DRAWINGS_ATM_CURRENT,749816,19.524872
CNT_DRAWINGS_POS_CURRENT,749816,19.524872
CNT_DRAWINGS_OTHER_CURRENT,749816,19.524872
AMT_INST_MIN_REGULARITY,305236,7.948208
CNT_INSTALMENT_MATURE_CUM,305236,7.948208



========== STATUS CREDIT ==========
NAME_CONTRACT_STATUS
Active           3698436
Completed         128918
Signed             11058
Demand              1365
Sent proposal        513
Refused               17
Approved               5
Name: count, dtype: int64

Traitement des valeurs manquantes...

Valeurs manquantes après traitement :
0

Création des variables métier...

========== TABLE FINALE ==========
(103558, 13)

Valeurs manquantes :
0

Aperçu :


,SK_ID_CURR,NB_CREDIT_CARD_RECORDS,NB_CREDIT_CARDS,AVG_BALANCE,MAX_BALANCE,AVG_CREDIT_LIMIT,AVG_PAYMENT,TOTAL_PAYMENT,NB_DELAY_MONTHS,MAX_DPD,MAX_DPD_DEF,CREDIT_UTILIZATION_RATIO,DPD_RATIO
0,100006,6,1,0.000000,0.00,270000.000000,0.000000,0.000,0,0,0,0.000000,0.000000
1,100011,74,1,54482.111149,189000.00,164189.189189,4520.067568,334485.000,0,0,0,0.331825,0.000000
2,100013,96,1,18159.919219,161420.22,131718.750000,6817.172344,654448.545,1,1,1,0.137869,0.010417
3,100021,17,1,0.000000,0.00,675000.000000,0.000000,0.000,0,0,0,0.000000,0.000000
4,100023,8,1,0.000000,0.00,135000.000000,0.000000,0.000,0,0,0,0.000000,0.000000



Fichier sauvegardé : credit_card_features.csv


# Transformation de la table `credit_card_balance` au niveau client

## Objectif

La table `credit_card_balance` contient l'historique mensuel des cartes de crédit des clients.

Chaque ligne représente un état mensuel d'une carte de crédit (`SK_ID_PREV`) et non un client. Un même client peut posséder plusieurs cartes et chaque carte peut être observée sur plusieurs mois.

L'objectif est de transformer cette table afin d'obtenir une seule ligne par client (`SK_ID_CURR`) contenant des indicateurs synthétiques décrivant son comportement d'utilisation des cartes de crédit.

---

# Analyse initiale

La table contient :

- **3 840 312 lignes**
- **23 variables**

Les principales variables décrivent :

- le solde de la carte (`AMT_BALANCE`) ;
- la limite de crédit (`AMT_CREDIT_LIMIT_ACTUAL`) ;
- les montants retirés ;
- les paiements effectués ;
- les montants restant dus ;
- les retards de paiement (`SK_DPD`, `SK_DPD_DEF`) ;
- le statut de la carte (`NAME_CONTRACT_STATUS`).

---

# Traitement des valeurs manquantes

Certaines variables présentent des valeurs manquantes.

Les variables correspondant à des opérations financières (paiements, retraits, retards) ont été remplacées par **0** lorsque l'absence de valeur signifie qu'aucune opération n'a été réalisée.

Les autres variables numériques ont été imputées par leur médiane afin de limiter l'influence des valeurs extrêmes.

À l'issue du traitement :

- Valeurs manquantes restantes : **0**

---

# Création des variables comportementales

Les informations mensuelles sont agrégées par client (`SK_ID_CURR`).

Les principales variables créées sont :

- **NB_CREDIT_CARD_RECORDS** : nombre total d'enregistrements mensuels ;
- **NB_CREDIT_CARDS** : nombre de cartes de crédit ;
- **AVG_BALANCE** : solde moyen ;
- **MAX_BALANCE** : solde maximal observé ;
- **AVG_CREDIT_LIMIT** : limite moyenne de crédit ;
- **AVG_PAYMENT** : paiement moyen ;
- **TOTAL_PAYMENT** : montant total payé ;
- **NB_DELAY_MONTHS** : nombre de mois avec retard ;
- **MAX_DPD** : retard maximal ;
- **MAX_DPD_DEF** : retard maximal en défaut ;
- **CREDIT_UTILIZATION_RATIO** : taux moyen d'utilisation de la limite de crédit ;
- **DPD_RATIO** : proportion de mois présentant un retard.

---

# Résultat obtenu

Après agrégation :

- une seule ligne représente chaque client ;
- la table finale contient **103 558 clients** ;
- **13 variables** décrivant le comportement des cartes de crédit sont obtenues ;
- aucune valeur manquante ne subsiste.

Le fichier final est enregistré sous le nom :

`credit_card_features.csv`

Cette table sera ensuite fusionnée avec les autres tables agrégées afin de construire le jeu de données final destiné au modèle de prédiction du risque de défaut.